In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter, CharacterTextSplitter

In [5]:
# Load PDF
loader = PyPDFLoader("Downloads/Data-Science-Projects/Legal-RAG/IPC.pdf")
pages = loader.load()
print(f"Total pages loaded: {len(pages)}")

Total pages loaded: 119


In [6]:
# Split into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
docs = splitter.split_documents(pages)
print(f"Total chunks created: {len(docs)}")
print(f"\nSample chunk:\n{docs[0].page_content}")

Total chunks created: 1146

Sample chunk:
1 
 
THE INDIAN PENAL CODE 
___________ 
ARRANGEMENT OF SECTIONS 
__________ 
CHAPTER I 
INTRODUCTION 
PREAMBLE 
SECTIONS 
1. Title and extent of operation of the Code.  
2. Punishment of offences committed within India.  
3. Punishment of offences committed beyond, but which by law may be tried within, India. 
4. Extension of Code to extra-territorial offences. 
5. Certain laws not to be affected by this Act. 
CHAPTER II 
GENERAL EXPLANATIONS


In [7]:
# Creating Vector Store + Embeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# Loading embedding model (converting text > vectors)
embeddings = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')

# store in ChromaDB (Local vector database)
vectorstore = Chroma.from_documents(docs, embeddings)

print(f'Vector store created with {len(docs)} documents!')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vector store created with 1146 documents!


In [8]:
# Groq LLM Setup
from langchain_groq import ChatGroq
import os

os.environ['GROQ_API_KEY'] = 'your_groq_api_key_here' # Your's Groq API Key

llm = ChatGroq(
    model_name = 'llama-3.3-70b-versatile',
    temperature = 0
)
print('Groq LLM Ready!')

Groq LLM Ready!


In [9]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

prompt = ChatPromptTemplate.from_template("""
You are an expert on Indian Penal Code (IPC).
Use the following IPC sections to answer the question.
If the answer is not in the context, say "I could not find this in the IPC."

Context: {context}
Question: {question}

Answer:""")

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

qa_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG Chain ready!")

RAG Chain ready!


In [10]:
# Asking a Legal question
questions = [
    'What is the punishment for murder?',
    'What is IPC section for kidnapping?',
    'What happens if someone causes arson?'
]

for q in questions:
    print(f'Q: {q}')
    print(f'A: {qa_chain.invoke(q)}')
    print('-' * 60)

Q: What is the punishment for murder?
A: The punishment for murder is death, or imprisonment for life, and the person shall also be liable to fine, as per Section 302 of the Indian Penal Code.
------------------------------------------------------------
Q: What is IPC section for kidnapping?
A: The IPC sections for kidnapping are:

- Section 359: Kidnapping (defines the two kinds of kidnapping: kidnapping from India and kidnapping from lawful guardianship)
- Section 360: Kidnapping from India (specifically deals with conveying a person beyond the limits of India without consent)
- Section 365: Kidnapping or abducting with intent secretly and wrongfully to confine person
- Section 366: Kidnapping, abducting or inducing woman to compel her marriage, etc.
- Section 367: Kidnapping or abducting in order to subject person to grievous hurt, slavery, etc.
- Section 368: Wrongfully concealing or keeping in confinement, kidnapped or abducted person
- Section 364A: Kidnapping for ransom, etc.
--